# Final Project — Algorithm Lab: A Command-Line Toolkit

Every previous lesson taught one tool at a time: a loop here, a dictionary there, a way to open a file. This final project asks you to combine all of them into one real program: **Algorithm Lab**, a command-line toolkit that runs classic algorithms on numbers, text, and lists, remembers what you did by writing to a file, and uses Python's standard library instead of reinventing everything from scratch.

## What you'll build

| Topic | Where you'll use it |
|---|---|
| Variables, data types, f-strings | Everywhere — every result you print |
| Operators, `input()`, type conversion | Every function that reads a number |
| `if` / `elif` / `else` | Prime/palindrome/anagram checks, validation |
| `while` loops | GCD, digit sum, number reversal, Collatz steps |
| `for` loops, `range()` | Prime checking, statistics, building lists |
| Lists: indexing, methods, `enumerate()`, `.copy()` | Search, sort, duplicate detection |
| Strings: slicing, immutability, methods, `ord()`/`chr()` | Palindrome check, Caesar cipher, word frequency |
| Functions: parameters, `return`, defaults | Every algorithm below is a function you write |
| `try`/`except`, dictionaries | Statistics, word counting |
| File I/O: `open()`, `with`, modes `"w"`/`"r"`/`"a"` | A run-history log and dataset save/load |
| Modules: `random`, `math`, `datetime` | Test data, cross-checking results, timestamps |

## How grading works in this notebook

- Every exercise gives you a function **signature** and **hints** — you write the body.
- Right below it is a **checker cell**. Run it after your own cell; it prints `✅ Correct!` or `❌ Incorrect`.
- **Don't edit the checker cells.** If you get `❌`, fix your function above and re-run both cells in order.
- A couple of exercises (random data, file logging) can't be checked against one fixed answer, since the result depends on randomness or the current time — those checkers instead verify that your function *behaves correctly* (right length, right format, right round-trip), the same idea used in Homework 1 for `input()`-based exercises.

Run the cell below first — it's the same grading helper you've already used in Homework 1 and 2.

In [ ]:
def check(condition, ok="Correct! Well done.", no="Incorrect. Check your logic above and try again."):
    if condition:
        print("\u2705", ok)
    else:
        print("\u274c", no)

In [ ]:
import random
import math
import datetime

---
## Section A — Number Algorithms

Four classic algorithms, plus a "compute statistics by hand" function. Each one must **return** a value instead of just printing — that's what lets the checkers test them, and what will let the finished menu program (Section F) reuse them later.

### A1 — Prime Checker

A number is prime if nothing between `2` and itself divides it evenly. You only need to check up to its square root — if a factor larger than the square root existed, its matching pair would have to be smaller than the square root, so you'd already have found it.

In [ ]:
def is_prime(n):
    # a number smaller than 2 is never prime
    if n < 2:
        return False

    # check whether anything from 2 up to int(math.sqrt(n)) (inclusive) divides n evenly
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False

    return True

In [ ]:
check(
    is_prime(2) is True
    and is_prime(1) is False
    and is_prime(0) is False
    and is_prime(97) is True
    and is_prime(100) is False
)

### A2 — Greatest Common Divisor (Euclidean Algorithm)

The Euclidean Algorithm is one of the oldest algorithms in history — Euclid described it over 2000 years ago. The idea: the GCD of two numbers doesn't change if you replace the larger one with the remainder of dividing it by the smaller one. Repeat until the remainder hits zero — whatever's left is the GCD.

In [ ]:
def gcd_euclid(a, b):
    # while b is not 0, replace (a, b) with (b, a % b)
    while b != 0:
        a, b = b, a % b

    return a

In [ ]:
check(gcd_euclid(462, 1071) == 21 and gcd_euclid(48, 18) == math.gcd(48, 18))

### A3 — Digit Sum & Reversed Number

Both use the same `while` loop trick: `n % 10` peels off the last digit, `n // 10` drops it. Make both work for a number of any length, not just a fixed size — and think about what should happen to trailing zeros when you reverse a number like `8300`.

In [ ]:
def digit_sum(n):
    n = abs(n)
    total = 0
    while n > 0:
        total += n % 10
        n = n // 10

    return total


def reverse_number(n):
    is_negative = n < 0
    n = abs(n)

    reversed_n = 0
    while n > 0:
        reversed_n = reversed_n * 10 + (n % 10)
        n = n // 10

    if is_negative:
        reversed_n = -reversed_n

    return reversed_n

In [ ]:
check(
    digit_sum(738295) == 34
    and reverse_number(8300) == 38
    and reverse_number(-120) == -21
)

### A4 — Collatz Conjecture Steps

Pick a number. If it's even, divide it by two. If it's odd, triple it and add one. Repeat. Nobody has ever found a starting number that doesn't eventually reach 1 — but nobody has *proven* it always will, either. It's one of math's most famous open problems, and the algorithm itself is only a few lines of code.

In [ ]:
def collatz_steps(n):
    steps = 0
    while n != 1:
        if n % 2 == 0:
            n = n // 2
        else:
            n = n * 3 + 1
        steps += 1

    return steps

In [ ]:
check(collatz_steps(6) == 8 and collatz_steps(27) == 111)

### A5 — List Statistics (By Hand)

Compute the minimum, maximum, sum, and mean of a list yourself, with a single `for` loop — the same "walk through and keep the best so far" pattern used for finding the biggest element in a list. Don't use `min()`, `max()`, or `sum()` inside your function; the checker uses them separately to confirm your hand-written version agrees.

In [ ]:
def list_statistics(numbers):
    if len(numbers) == 0:
        return None

    smallest = largest = numbers[0]
    total = 0
    for number in numbers:
        if number < smallest:
            smallest = number
        if number > largest:
            largest = number
        total += number

    mean = total / len(numbers)

    return {"min": smallest, "max": largest, "sum": total, "mean": mean}

In [ ]:
grades = [70, 85, 90, 60, 100, 78, 92]
stats = list_statistics(grades)
check(
    stats is not None
    and stats["min"] == min(grades)
    and stats["max"] == max(grades)
    and stats["sum"] == sum(grades)
    and abs(stats["mean"] - sum(grades) / len(grades)) < 1e-9
)

---
## Section B — Text Algorithms

Strings behave a lot like lists of characters (Lesson 8), so several list ideas — slicing, looping, building something new instead of mutating — carry straight over.

### B1 — Palindrome Checker

A palindrome reads the same forwards and backwards. Normalize the text first (lowercase, no spaces) so phrases like *"A man a plan a canal Panama"* count too, not just single words.

In [ ]:
def is_palindrome(text):
    cleaned = text.lower().replace(" ", "")

    return cleaned == cleaned[::-1]

In [ ]:
check(
    is_palindrome("kayak") is True
    and is_palindrome("algorithm") is False
    and is_palindrome("A man a plan a canal Panama") is True
)

### B2 — Caesar Cipher (Fixed & Complete)

Lesson 8 built a first version of the Caesar cipher with `ord()`/`chr()`, but flagged two problems: it didn't wrap around past `Z`, and it broke on anything that wasn't a letter. Fix both here. A `decode=True` flag should undo an encode with the same shift — think about what that means for the direction of the shift.

In [ ]:
def caesar_cipher(text, shift=3, decode=False):
    if decode:
        shift = -shift

    result = ""
    for ch in text:
        if ch.isupper():
            result += chr((ord(ch) - ord("A") + shift) % 26 + ord("A"))
        elif ch.islower():
            result += chr((ord(ch) - ord("a") + shift) % 26 + ord("a"))
        else:
            result += ch

    return result

In [ ]:
encoded = caesar_cipher("Hello, World!", shift=3)
check(
    encoded == "Khoor, Zruog!"
    and caesar_cipher(encoded, shift=3, decode=True) == "Hello, World!"
    and caesar_cipher("XYZ", shift=3) == "ABC"   # this is exactly what Lesson 8's version got wrong
)

### B3 — Word Frequency Counter

A dictionary is the natural fit: each unique word is a key, and the value is how many times it showed up. Lowercase everything first (so `"The"` and `"the"` count as the same word) and strip a small set of punctuation characters off each word.

In [ ]:
def word_frequency(text):
    words = text.lower().split()

    frequency = {}
    for word in words:
        word = word.strip(".,!?;:\"'()")
        if word in frequency:
            frequency[word] += 1
        else:
            frequency[word] = 1

    return frequency

In [ ]:
freq = word_frequency("The quick brown fox jumps over the lazy dog. The dog barks, but the fox runs on.")
check(freq.get("the") == 4 and freq.get("fox") == 2 and freq.get("dog") == 2)

### B4 — Anagram Checker

Two words are anagrams if they're made of exactly the same letters, just rearranged. Instead of counting letters by hand, reuse a Lesson 6 tool in a new way: `sorted()` on a string gives back a sorted **list** of its characters.

In [ ]:
def is_anagram(word1, word2):
    cleaned1 = word1.lower().replace(" ", "")
    cleaned2 = word2.lower().replace(" ", "")

    return sorted(cleaned1) == sorted(cleaned2)

In [ ]:
check(
    is_anagram("listen", "silent") is True
    and is_anagram("Dormitory", "Dirty Room") is True
    and is_anagram("hello", "world") is False
)

---
## Section C — List & Data Algorithms

Search, sort, and duplicate detection — three of the most common tasks you'll ever run on a list, built from scratch instead of relying on `in`, `sorted()`, or `set()`.

### C1 — Linear Search

Walk through the list one item at a time. If you finish the loop without finding the target, return `-1` — a common convention for "not found" borrowed from real search functions.

In [ ]:
def linear_search(data, target):
    for index, value in enumerate(data):
        if value == target:
            return index

    return -1

In [ ]:
check(
    linear_search([4, 8, 15, 16, 23, 42], 23) == 4
    and linear_search([4, 8, 15, 16, 23, 42], 99) == -1
)

### C2 — Bubble Sort

Bubble sort repeatedly walks through the list, swapping any neighbors that are in the wrong order. Each full pass "bubbles" the largest remaining value to the end, so after enough passes the whole list is sorted.

Start with `result = data.copy()` — that's the Lesson 6 copy trap. Without it, sorting `result` would silently sort the caller's original list too.

In [ ]:
def bubble_sort(data):
    result = data.copy()
    n = len(result)

    for i in range(n):
        for j in range(n - 1 - i):
            if result[j] > result[j + 1]:
                result[j], result[j + 1] = result[j + 1], result[j]

    return result

In [ ]:
original = [64, 34, 25, 12, 22, 11, 90]
result = bubble_sort(original)
check(result == sorted(original) and original == [64, 34, 25, 12, 22, 11, 90])

### C3 — Find Duplicates

Track how many times each value has been seen so far with a dictionary. The moment a value is seen for the *second* time, record it — so each duplicate value ends up in the result only once, no matter how many extra times it repeats.

In [ ]:
def find_duplicates(data):
    seen_counts = {}
    duplicates = []

    for value in data:
        if value in seen_counts:
            seen_counts[value] += 1
            if seen_counts[value] == 2:
                duplicates.append(value)
        else:
            seen_counts[value] = 1

    return duplicates

In [ ]:
check(sorted(find_duplicates([3, 7, 2, 7, 5, 3, 3, 9])) == [3, 7])

### C4 — Random Test Data

Rather than always typing test numbers by hand, the `random` library can generate them for you — handy for trying the algorithms above on data you haven't seen in advance. Since the result is random, the checker can't compare against one fixed list — instead it checks that your function *behaves correctly*: the right amount of numbers, each one inside the requested range.

In [ ]:
def generate_random_list(size=10, low=1, high=100):
    result = []
    for i in range(size):
        result.append(random.randint(low, high))

    return result

In [ ]:
sample = generate_random_list(size=15, low=1, high=50)
check(len(sample) == 15 and all(1 <= x <= 50 for x in sample))

---
## Section D — Saving Results to Disk

Every algorithm above forgets its results the moment the program ends. Real tools don't — they write to disk. Build a small logging system (Lesson 12's `open()`/`with`/append mode) combined with `datetime` (Lesson 13) so every entry is timestamped, plus a way to save and reload a dataset.

In [ ]:
LOG_FILE = "algorithm_lab_log.txt"
DATA_FILE = "algorithm_lab_dataset.txt"

### D1 — Logging Every Run

Each log line should look like:

```
[2026-07-25 14:32:10] PRIME_CHECK | n=97 -> True
```

In [ ]:
def log_result(action, details, filename=LOG_FILE):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(filename, "a") as file:
        file.write(f"[{timestamp}] {action} | {details}\n")

In [ ]:
# This checker uses its own file (test_log.txt) so it doesn't depend on what's already
# in algorithm_lab_log.txt from earlier runs.
with open("test_log.txt", "w") as file:
    pass  # start from an empty file

log_result("PRIME_CHECK", "n=97 -> True", filename="test_log.txt")
log_result("GCD", "a=462, b=1071 -> 21", filename="test_log.txt")

with open("test_log.txt", "r") as file:
    lines = file.readlines()

check(
    len(lines) == 2
    and lines[0].startswith("[") and "PRIME_CHECK" in lines[0]
    and lines[1].startswith("[") and "GCD" in lines[1]
)

### D2 — Reading The History Back

`view_history()` should print the whole log file. If the file doesn't exist yet (nobody has logged anything), handle that gracefully instead of crashing — Lesson 11's `try`/`except` is exactly for this.

In [ ]:
def view_history(filename=LOG_FILE):
    try:
        with open(filename, "r") as file:
            content = file.read()
    except FileNotFoundError:
        print("No history yet.")
        return

    if content == "":
        print("No history yet.")
    else:
        print(content)


def clear_history(filename=LOG_FILE):
    with open(filename, "w") as file:
        pass

In [ ]:
clear_history(filename="test_log.txt")
with open("test_log.txt", "r") as file:
    content_after_clear = file.read()

log_result("PRIME_CHECK", "n=13 -> True", filename="test_log.txt")

import io
import contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    view_history(filename="test_log.txt")
printed = buf.getvalue()

check(content_after_clear == "" and "PRIME_CHECK" in printed)

### D3 — Saving & Loading a Dataset

Besides logging *what happened*, you can persist actual data too — like a list of numbers — so it survives between runs. Join the list into one comma-separated line to write, and split it back apart (converting each piece from `str` back to `int`, since everything read from a file comes back as text) when loading.

In [ ]:
def save_dataset(data, filename=DATA_FILE):
    with open(filename, "w") as file:
        file.write(",".join(str(x) for x in data))


def load_dataset(filename=DATA_FILE):
    try:
        with open(filename, "r") as file:
            content = file.read()

        pieces = content.split(",")
        result = []
        for piece in pieces:
            result.append(int(piece))

        return result
    except FileNotFoundError:
        return []
    except ValueError:
        return []

In [ ]:
save_dataset([5, 3, 8, 1, 9], filename="test_dataset.txt")
loaded_back = load_dataset(filename="test_dataset.txt")
check(loaded_back == [5, 3, 8, 1, 9])

---
## Section E — Safe Input Helpers

`int(input(...))` crashes the whole program the instant someone types something that isn't a number. Lesson 11's `try`/`except` fixes that — but instead of wrapping every single `input()` call in the menu (Section F) with its own `try`/`except`, write the pattern **once** as a reusable function.

The checkers below can't ask you to type anything by hand, so they temporarily replace Python's `input()` with a fake version that returns preset answers — you don't need to understand how that trick works, just that it lets your function be tested automatically.

In [ ]:
def get_int(prompt):
    while True:
        try:
            return int(input(prompt))
        except ValueError:
            print("That's not a whole number. Please try again.")


def get_int_list(prompt):
    while True:
        try:
            raw = input(prompt)
            pieces = raw.split(",")
            numbers = []
            for piece in pieces:
                numbers.append(int(piece.strip()))
            return numbers
        except ValueError:
            print("That's not a valid list of numbers. Please try again.")

In [ ]:
import builtins

def _check_get_int():
    original_input = builtins.input
    responses = iter(["not a number", "42"])
    builtins.input = lambda prompt="": next(responses)
    try:
        return get_int("test: ") == 42
    finally:
        builtins.input = original_input

def _check_get_int_list():
    original_input = builtins.input
    builtins.input = lambda prompt="": "5, 3, 8, 1"
    try:
        return get_int_list("test: ") == [5, 3, 8, 1]
    finally:
        builtins.input = original_input

check(_check_get_int() and _check_get_int_list())

---
## Section F — The Algorithm Lab Menu (Provided)

Everything above was your work. This last part is **given to you**, the same way Lesson 15's final menu was given complete — wiring a menu together is just a bigger version of the `if`/`elif` decisions you've already made since Lesson 3, so there's nothing new to practice here. What *is* new: this program only works once every function in Sections A–E is correctly implemented, since it calls all of them directly.

Run the cells below top to bottom once (they just define the menu functions — nothing happens yet), then run the very last cell to actually play with Algorithm Lab.

In [ ]:
def print_main_menu():
    print("\n" + "=" * 42)
    print("          ALGORITHM LAB \u2014 MAIN MENU")
    print("=" * 42)
    print("1) Number Algorithms")
    print("2) Text Algorithms")
    print("3) List / Data Algorithms")
    print("4) File & History")
    print("5) Exit")


def run_number_menu():
    print("\n--- Number Algorithms ---")
    print("1) Prime checker")
    print("2) Greatest Common Divisor (Euclidean Algorithm)")
    print("3) Digit sum & reversed number")
    print("4) Collatz conjecture step count")
    print("5) List statistics (min / max / mean)")
    print("6) Back to main menu")
    choice = input("Choose an option (1-6): ")

    if choice == "1":
        n = get_int("Enter a whole number: ")
        result = is_prime(n)
        print(f"{n} is {'a prime number' if result else 'not a prime number'}.")
        log_result("PRIME_CHECK", f"n={n} -> {result}")
    elif choice == "2":
        x = get_int("Enter the first number: ")
        y = get_int("Enter the second number: ")
        result = gcd_euclid(x, y)
        print(f"GCD({x}, {y}) = {result}  (math.gcd agrees: {math.gcd(x, y)})")
        log_result("GCD", f"a={x}, b={y} -> {result}")
    elif choice == "3":
        n = get_int("Enter a whole number: ")
        print(f"Digit sum: {digit_sum(n)}")
        print(f"Reversed: {reverse_number(n)}")
        log_result("DIGIT_OPS", f"n={n} -> sum={digit_sum(n)}, reversed={reverse_number(n)}")
    elif choice == "4":
        n = get_int("Enter a starting number: ")
        steps = collatz_steps(n)
        print(f"It takes {steps} steps for {n} to reach 1.")
        log_result("COLLATZ", f"start={n} -> steps={steps}")
    elif choice == "5":
        numbers = get_int_list("Enter numbers separated by commas: ")
        stats = list_statistics(numbers)
        print(f"min={stats['min']}, max={stats['max']}, sum={stats['sum']}, mean={stats['mean']:.2f}")
        log_result("LIST_STATS", f"data={numbers} -> {stats}")
    elif choice == "6":
        return
    else:
        print("Invalid option, please try again.")


def run_text_menu():
    print("\n--- Text Algorithms ---")
    print("1) Palindrome checker")
    print("2) Caesar cipher (encode/decode)")
    print("3) Word frequency counter")
    print("4) Anagram checker")
    print("5) Back to main menu")
    choice = input("Choose an option (1-5): ")

    if choice == "1":
        text = input("Enter a word or phrase: ")
        result = is_palindrome(text)
        print(f'"{text}" {"IS" if result else "is NOT"} a palindrome.')
        log_result("PALINDROME", f"text={text!r} -> {result}")
    elif choice == "2":
        text = input("Enter your message: ")
        shift = get_int("Enter a shift value (e.g. 3): ")
        mode = input("Type 'e' to encode or 'd' to decode: ").strip().lower()
        decode = mode == "d"
        result = caesar_cipher(text, shift=shift, decode=decode)
        print("Result:", result)
        log_result("CAESAR", f"text={text!r} shift={shift} decode={decode} -> {result!r}")
    elif choice == "3":
        text = input("Enter a sentence or paragraph: ")
        frequency = word_frequency(text)
        print(frequency)
        if frequency:
            most_common = max(frequency, key=frequency.get)
            print(f"Most common word: '{most_common}' ({frequency[most_common]} times)")
        log_result("WORD_FREQ", f"text={text!r} -> {frequency}")
    elif choice == "4":
        w1 = input("Enter the first word: ")
        w2 = input("Enter the second word: ")
        result = is_anagram(w1, w2)
        print(f'"{w1}" and "{w2}" {"ARE" if result else "are NOT"} anagrams.')
        log_result("ANAGRAM", f"w1={w1!r} w2={w2!r} -> {result}")
    elif choice == "5":
        return
    else:
        print("Invalid option, please try again.")


def run_data_menu():
    print("\n--- List / Data Algorithms ---")
    print("1) Linear search")
    print("2) Bubble sort (compared with built-in sorted())")
    print("3) Find duplicates")
    print("4) Generate a random dataset")
    print("5) Back to main menu")
    choice = input("Choose an option (1-5): ")

    if choice in ("1", "2", "3"):
        source = input("Use your own numbers or a random dataset? (own/random): ").strip().lower()
        if source == "random":
            data = generate_random_list()
            print("Generated:", data)
        else:
            data = get_int_list("Enter numbers separated by commas: ")

        if choice == "1":
            target = get_int("Enter the number to search for: ")
            index = linear_search(data, target)
            print(f"Found at index {index}." if index != -1 else "Not found.")
            log_result("SEARCH", f"data={data} target={target} -> index={index}")
        elif choice == "2":
            result = bubble_sort(data)
            print("Sorted:", result)
            print("Matches built-in sorted():", result == sorted(data))
            log_result("SORT", f"before={data} after={result}")
        elif choice == "3":
            dupes = find_duplicates(data)
            print("Duplicates:", dupes if dupes else "none found")
            log_result("DUPLICATES", f"data={data} -> {dupes}")
    elif choice == "4":
        size = get_int("How many numbers? ")
        data = generate_random_list(size=size)
        print("Random dataset:", data)
        log_result("RANDOM_DATA", f"size={size} -> {data}")
    elif choice == "5":
        return
    else:
        print("Invalid option, please try again.")


def run_file_menu():
    print("\n--- File & History ---")
    print("1) View run history")
    print("2) Clear run history")
    print("3) Save a dataset to file")
    print("4) Load a dataset from file")
    print("5) Back to main menu")
    choice = input("Choose an option (1-5): ")

    if choice == "1":
        view_history()
    elif choice == "2":
        clear_history()
    elif choice == "3":
        data = get_int_list("Enter numbers separated by commas: ")
        save_dataset(data)
    elif choice == "4":
        data = load_dataset()
        print("Loaded dataset:", data)
    elif choice == "5":
        return
    else:
        print("Invalid option, please try again.")

### Run It!

This keeps asking for input until you choose **Exit**. Run this cell yourself once your functions in Sections A–E are all passing their checkers.

In [ ]:
print("Welcome to Algorithm Lab! Every action you take gets logged to algorithm_lab_log.txt.")

while True:
    print_main_menu()
    choice = input("Choose an option (1-5): ")

    if choice == "1":
        run_number_menu()
    elif choice == "2":
        run_text_menu()
    elif choice == "3":
        run_data_menu()
    elif choice == "4":
        run_file_menu()
    elif choice == "5":
        print("Goodbye! Your run history has been saved to algorithm_lab_log.txt.")
        break
    else:
        print("Invalid option, please try again.")

---
## Wrap-Up

Once every checker above prints `✅ Correct!` and the Section F menu runs end to end, you'll have built a single program that makes decisions, repeats work with both kinds of loops, stores data in lists and dictionaries, manipulates text, organizes logic into reusable functions, reads and writes files with real timestamps, and leans on the standard library instead of reinventing everything. That's the same toolkit real, working software is built from — the difference between this and a professional command-line tool is scale, not kind.

## Going Further (Optional)

- **Binary search**: now that you can sort a list, write a binary search that only works on sorted data and compare how many steps it needs versus `linear_search` on a large random list.
- **Recursion**: rewrite `gcd_euclid` (or a factorial function) using recursion instead of a `while` loop.
- **JSON storage**: look up Python's `json` module and rewrite `save_dataset`/`load_dataset` to store structured data instead of a comma-separated string.
- **A leaderboard**: extend the file logger to track the largest number successfully prime-checked, or the longest Collatz chain found so far.
- **A second sorting algorithm**: implement selection sort or insertion sort and count how many comparisons/swaps each one makes versus `bubble_sort` on the same data.